# Fine-Tuning: embedding-gemma-300M on TripLegal-CL


**Objective.** The central claim of the paper is that TripLegal-CL provides
an effective contrastive training signal for adapting dense bi-encoder
retrievers to the Spanish legal domain. To validate this, we fine-tune
`google/embeddinggemma-300m` — a lightweight embedding model derived
from the Gemma LLM family — using contrastive learning on TripLegal-CL.
If the fine-tuned model consistently outperforms its baseline across
all IR metrics, this confirms that the corpus provides useful
domain-specific supervision.

**Dev evaluator.** During training, a dev evaluator (50K queries, 80K
corpus) runs every 100 steps to monitor convergence. It is drawn from
the **training region** (first 380K instances) and is intentionally
smaller for speed.

## 1. Environment Setup

Embedding-Gemma requires a specific `transformers` branch for
compatibility.

In [1]:
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview


  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-v7_vii26
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-v7_vii26
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.1 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-4.57.0.dev0-py3-none-any.whl size=12604658 sha256=39e244371510a230a8feb51d060d2ae1641eccb975894db73ed4a7402e3ce378
  Stored in directory: /tmp/pip-ephem-wheel-cache-45t1s86j/wheels/3a/21/76/c31899bac2cf601d3c74091b26a413bc3fb54770d5ccb5c924
Successfully built transformers
  Attempting uni

## 2. Imports


In [2]:
import logging
import traceback

from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

## 3. Load Model


In [3]:
model = SentenceTransformer(
    "google/embeddinggemma-300m",
    model_card_data=SentenceTransformerModelCardData(
        language="es",
        license="apache-2.0",
        model_name="EmbeddingGemma-300m trained on TripLegal-CL Legal Spanish.",
    ),
)

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]




**Note on Gemma prefixes:** EmbeddingGemma uses instructional prompts
that are prepended to the input text. According to the
[model card](https://huggingface.co/google/embeddinggemma-300m), query
prompts follow the form `task: {description} | query:` and document
prompts follow the form `title: {title} | text:`. In sentence-transformers,
these are registered in
[`config_sentence_transformers.json`](https://huggingface.co/google/embeddinggemma-300m/blob/main/config_sentence_transformers.json)
under the keys `"query"` and `"document"` (which correspond to the
task names `"Retrieval-query"` and `"Retrieval-document"` in the model
card). Using `model.prompts["query"]` and `model.prompts["document"]`
resolves to the correct full prompt strings automatically.

> **⚠ FP16 warning:** The [model card](https://huggingface.co/google/embeddinggemma-300m)
> states: *"EmbeddingGemma activations do not support float16."*
> If you encounter numerical issues with `fp16=True`, switch to
> `fp16=False, bf16=True` (requires Ampere GPU or newer) or disable
> mixed precision entirely.



## 4. Prepare Data Helper


In [4]:
def select_first_pos(example):
    if example["pos"]:
        return {"query": example["query"], "pos": example["pos"][0]}

## 5. Load and Split Dataset


In [5]:
SEED = 42
TRAIN_N = 300_000
EVAL_N  = 50_000
TEST_N  = 30_000

base = load_dataset("wilfredomartel/TripLegal-CL", split="train").shuffle(seed=SEED)

# Disjoint ranges — no data leakage
train_dataset = base.select(range(0, TRAIN_N))
eval_dataset  = base.select(range(TRAIN_N, TRAIN_N + EVAL_N))
test_dataset  = base.select(range(TRAIN_N + EVAL_N, TRAIN_N + EVAL_N + TEST_N))

cols_to_remove = ["neg", "pos_score", "neg_score"]

train_dataset = train_dataset.remove_columns(cols_to_remove).map(select_first_pos)
eval_dataset  = eval_dataset.remove_columns(cols_to_remove).map(select_first_pos)
test_dataset  = test_dataset.remove_columns(cols_to_remove).map(select_first_pos)

print(f"Train: {len(train_dataset):,}")
print(f"Eval:  {len(eval_dataset):,}")
print(f"Test:  {len(test_dataset):,}")
print(train_dataset[0])

README.md:   0%|          | 0.00/428 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/206M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/592382 [00:00<?, ? examples/s]

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Train: 300,000
Eval:  50,000
Test:  30,000
{'query': '¿Por qué la Primera Sala de la Suprema Corte de Justicia de la Nación declaró infundado el recurso de reclamación 456/2019, confirmando el desechamiento del recurso de revisión?', 'pos': 'La Primera Sala de la Suprema Corte de Justicia de la Nación declaró infundado el recurso de reclamación 456/2019, confirmando el desechamiento del recurso de revisión, al determinar que los agravios presentados por Manuel Contreras Ramos no combatían las razones del acuerdo de presidencia recurrido. El acuerdo de desechamiento se basó en la inexistencia de una cuestión propiamente constitucional, mientras que los agravios del recurrente se enfocaron en demostrar la importancia y trascendencia del asunto, sin desvirtuar la falta de un tema de constitucionalidad. La Sala aplicó la tesis 1a. XXXVI/2018 (10a.) para señalar que los agravios que no desvirtúan la inexistencia de una cuestión constitucional son inoperantes, y que la falta de un tema de co

## 6. Loss Function and Training Arguments


In [6]:
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)

run_name = "GemmaEmbedding-TripLegalCL-300k"

args = SentenceTransformerTrainingArguments(
    output_dir=f"models/{run_name}",
    num_train_epochs=1,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    learning_rate=1.5e-5,
    warmup_ratio=0.03,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # Gemma built-in prompts — uses the model's own prefix templates
    prompts={
        "query": model.prompts["query"],
        "pos": model.prompts["document"],
    },
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=20,
    run_name=run_name,
)


**Note on `learning_rate=1.5e-5`:** Slightly lower than the 2e-5 used
for BGE-M3 and E5. Gemma-300M is a smaller model (~300M params vs ~560M)
and empirically benefits from a gentler learning rate to avoid
overshooting.

### How contrastive learning works in this training setup

> **Sources:** This explanation is based on the official `sentence-transformers`
> documentation and source code:
> - [MultipleNegativesRankingLoss — Losses documentation](https://sbert.net/docs/package_reference/sentence_transformer/losses.html)
> - [CachedMultipleNegativesRankingLoss — source code](https://github.com/huggingface/sentence-transformers/blob/main/sentence_transformers/losses/CachedMultipleNegativesRankingLoss.py)
> - [NoDuplicatesBatchSampler — Samplers documentation](https://sbert.net/docs/package_reference/sentence_transformer/sampler.html)
> - [Loss Overview — sentence-transformers](https://sbert.net/docs/sentence_transformer/loss_overview.html)
> - Original paper: *Efficient Natural Language Response Suggestion for Smart Reply*, Section 4.4 ([Henderson et al., 2017](https://huggingface.co/papers/1705.00652))

This section explains how the contrastive training signal is formed
during fine-tuning — the mechanism referred to as "contrastive learning"
in the paper.

**Training data format.** Each training example is a `(query, pos)` pair.
We do NOT explicitly provide negative passages to the trainer. Instead,
negatives are constructed automatically at training time through a
mechanism called **in-batch negatives**.

**In-batch negatives (InfoNCE / MNRL).** Given a batch of `B` pairs
`{(q₁, p₁), (q₂, p₂), ..., (qB, pB)}`, the loss function:

1. Encodes all `B` queries and all `B` passages into dense vectors.
2. Computes a `B × B` cosine similarity matrix between all queries
   and all passages.
3. For each query `qᵢ`, the passage `pᵢ` is the **positive** (correct
   answer), and all other passages `{p₁, ..., pᵢ₋₁, pᵢ₊₁, ..., pB}`
   are treated as **negatives** (wrong answers).
4. Applies cross-entropy loss to push `qᵢ` closer to `pᵢ` and away
   from all other passages in the batch.

```
Similarity matrix (batch_size=4 example):

              p₁     p₂     p₃     p₄
        q₁ [ 0.92   0.45   0.51   0.38 ]  ← maximize (q₁, p₁)
        q₂ [ 0.41   0.89   0.47   0.52 ]  ← maximize (q₂, p₂)
        q₃ [ 0.50   0.43   0.91   0.40 ]  ← maximize (q₃, p₃)
        q₄ [ 0.39   0.48   0.42   0.87 ]  ← maximize (q₄, p₄)

Diagonal = positive pairs (should be highest in each row)
Off-diagonal = in-batch negatives (should be lower)
```

This means that with a batch size of 128, each query has **127 implicit
negatives** per training step — all from the same legal domain, making
them naturally hard negatives.

**Why larger batches improve performance.** More samples per batch =
more in-batch negatives = harder contrastive signal = better
discrimination. This is why `CachedMultipleNegativesRankingLoss` is
valuable: it allows an effective batch size of 128 while only using
the GPU memory of `mini_batch_size=16`, by caching intermediate
embeddings (GradCache; Gao et al., 2021).

**Role of `BatchSamplers.NO_DUPLICATES`.** This sampler ensures that
no two samples in the same batch share identical text (query or
passage). This is critical because:

- If `p₃ == p₇` (duplicate passages in the batch), then `q₃` would
  have its own correct answer appearing as a "negative" — sending a
  contradictory gradient signal to the model.
- `NO_DUPLICATES` prevents this by checking for text duplicates when
  forming each batch.

**Important:** `NO_DUPLICATES` is a **sampler** (controls batch
composition), not a loss function. It does not generate negatives —
the loss function does that via the similarity matrix above.

**Summary of roles:**

| Component | Role |
|-----------|------|
| `CachedMNRL` (loss) | Constructs in-batch negatives from the B×B similarity matrix and computes cross-entropy |
| `NO_DUPLICATES` (sampler) | Ensures no duplicate texts in a batch, preventing false negatives |
| `per_device_train_batch_size=128` | Controls how many in-batch negatives each query sees (127) |
| `mini_batch_size=16` | Controls GPU memory usage (forward pass in chunks of 16) |


## 7. Build Dev Evaluator (monitoring during training)

This evaluator runs every 100 training steps to monitor convergence.
It uses data from the **training region** (first 380K instances), NOT
from the final evaluation region. It is intentionally smaller (50K
queries, 80K corpus) for speed.

In [7]:
queries = dict(enumerate(eval_dataset["query"]))

corpus_list = eval_dataset["pos"][:] + train_dataset.select(range(30_000))["pos"][:]
corpus = dict(enumerate(corpus_list))

relevant_docs = {idx: [idx] for idx in queries}

dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-spanish-gemma-eval-50kq-80kd",
    show_progress_bar=True,
)

# Evaluate base model before training
dev_evaluator(model)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [04:19<04:19, 259.83s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [06:57<00:00, 208.53s/it]


{'legal-spanish-gemma-eval-50kq-80kd_cosine_accuracy@1': 0.84898,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_accuracy@3': 0.91116,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_accuracy@5': 0.92742,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_accuracy@10': 0.94462,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_precision@1': 0.84898,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_precision@3': 0.30372,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_precision@5': 0.185484,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_precision@10': 0.094462,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_recall@1': 0.84898,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_recall@3': 0.91116,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_recall@5': 0.92742,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_recall@10': 0.94462,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_ndcg@10': 0.8983822243994213,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_mrr@10': 0.8834103650793612,
 'legal-spanish-gemma-eval-50kq-80kd_cosine_map@100': 0.8849

## 8. Train


In [8]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 ca806a58eb740a9937b1e3768432e38bd0043d6e


wandb: WARNING Invalid choice
wandb: Enter your choice:

 ca806a58eb740a9937b1e3768432e38bd0043d6e


...

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [02:01<02:01, 121.28s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [03:16<00:00, 98.01s/it]


TrainOutput(global_step=2344, training_loss=0.008920363484883408, metrics={'train_runtime': 27332.5315, 'train_samples_per_second': 10.976, 'train_steps_per_second': 0.086, 'total_flos': 0.0, 'train_loss': 0.008920363484883408, 'epoch': 1.0})

## 9. Post-Training Evaluation and Save

Running the evaluators after training serves two purposes: (1) confirm
final metrics, and (2) **log results into the model card** — Hugging Face
automatically includes the last evaluation scores in the model card
when pushing to the Hub.

We evaluate on both the **dev set** (50K queries) and a held-out **test
set** (30K queries, 20K corpus) to provide two independent performance
snapshots in the model card.

In [9]:
# Re-run dev evaluator — results are logged into the model card
dev_evaluator(model)

# Test set evaluation (30K queries, 50K corpus) — also logged into model card
queries = dict(enumerate(test_dataset["query"]))
corpus_list = test_dataset["pos"][:] + train_dataset.select(range(20_000))["pos"][:]
corpus = dict(enumerate(corpus_list))

relevant_docs = {idx: [idx] for idx in queries}
test_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-spanish-gemma-test-30kq-50kd",
    show_progress_bar=True,
)
test_evaluator(model)

# Save the trained model
final_output_dir = f"models/{run_name}/final"
model.save_pretrained(final_output_dir)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [01:58<01:58, 118.68s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [03:12<00:00, 96.07s/it]


Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [01:57<00:00, 117.85s/it]



> **Note:** The official results reported in **Table 4** of the paper come
> from a separate evaluation notebook using the `legalspanish-eval-60kq-120kd`
> benchmark (last 200K instances). The evaluations above are for the
> model card and internal validation only.

## 10. Push to Hub


In [10]:
try:
    model.push_to_hub(run_name)
except Exception:
    logging.error(
        f"Error uploading model to Hub:\n{traceback.format_exc()}"
        f"Model is saved locally at: {final_output_dir}\n"
        f"To retry: model = SentenceTransformer('{final_output_dir}')\n"
        f"Then: model.push_to_hub('{run_name}')"
    )

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpkdb8jfux/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...pkdb8jfux/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...db8jfux/model.safetensors:   0%|          |  556kB / 1.21GB            

  ...2_Dense/model.safetensors:   6%|6         |  592kB / 9.44MB            

  ...3_Dense/model.safetensors:   6%|6         |  592kB / 9.44MB            